# Early Dysgraphia Screening from Handwriting

## Overview
In this project, we are building an end-to-end computer vision pipeline to automate early dysgraphia screening using full-image handwriting scans. By evaluating 1,625 clinical handwriting records,the fine-tuned MobileNetV2 classification model successfully identifies at-risk children while minimizing false alarms.

## 1. Business Understanding

###  The Current Process and Limitations
Teachers and parents currently have no quick, low-cost way to check whether a child's handwriting shows dysgraphia-related patterns before committing to a slow, expensive specialist referral. Consequently, children are often mislabeled as slow or lazy, and detection depends entirely on someone noticing the issue over weeks or months. 

The key stakeholders are:
*   **At-Risk Children & Parents:** They need early detection to avoid the psychological damage of being mislabeled.
*   **Teachers & Under-resourced Schools:** They need a fast, low-cost screening aid that does not require an educational psychologist on staff.
*   **Specialists:** They need a reliable referral pipeline.

The implications of model errors are asymmetric. If we falsely flag a healthy student (False Positive), it causes temporary parental worry; however, if we miss a true case (False Negative), a child remains undiagnosed and unsupported. Therefore, this tool is strictly a binary screening and referral aid, not a definitive clinical diagnosis.

###  Modeling Goals and Metrics
The primary goal is to build a deep learning classifier that maximizes early detection without over-diagnosing healthy handwriting. Because the dataset is inherently imbalanced, standard accuracy is a dangerously inadequate metric.

*   **Primary Metric:** Recall on the Potential Dysgraphia class. Minimizing False Negatives is the absolute priority to ensure no at-risk child is missed.
*   **Secondary Metric:** ROC-AUC and Youden's J statistic to mathematically balance the decision threshold and maintain an acceptable precision rate.
*   **Baseline Target:** The transfer learning architecture must decisively outperform a custom 3-block baseline CNN trained from scratch.

## 2. Data Understanding

###  Exploratory Data Analysis
The Potential Dysgraphia Handwriting Dataset consists of 1,625 handwriting images. The initial audit reveals a strict class imbalance: roughly 64.5% (1,049) of the images are labeled 'Potential Dysgraphia', while only 35.5% (576) are 'Low Potential Dysgraphia'. 


###  Image Characteristics & Quality Checks
Unlike tabular data, the features are raw pixels. The primary challenge here is the extreme geometric variability of the clinical scans. 
*   **Dimensionality:** Image dimensions swing wildly from 506×55 px to 1318×365 px.
*   **Orientation & Margins:** Scans contain crooked handwriting lines and massive amounts of empty scanner margins.
*   **Content:** The handwriting is in Malay. However, dysgraphia is a motor and neurological disorder, not a linguistic one; the model will focus strictly on stroke mechanics, baseline alignment, and irregular spacing, rendering the specific language completely irrelevant.

In [ ]:
# imports
import os, hashlib, random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, f1_score, recall_score, precision_score,
)

# Set the random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASSES = ["Low Potential Dysgraphia", "Potential Dysgraphia"]

# Potential Dysgraphia encoded it as the positive class (1).
LABEL_MAP = {"Low Potential Dysgraphia": 0, "Potential Dysgraphia": 1}
POSITIVE_CLASS = "Potential Dysgraphia"

# Dataset root folder
DATASET_ROOT = Path("DATASET DYSGRAPHIA HANDWRITING")
assert DATASET_ROOT.exists(), (
    f"Could not find '{DATASET_ROOT}'. Update DATASET_ROOT to point at the "
    f"unzipped dataset folder (the one that directly contains the class subfolders)."
)

# width, height fed into preprocess_image
TARGET_SIZE = (256, 256)

2026-09-08 15:46:37.156669: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-08 15:46:38.036914: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-08 15:46:41.943696: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 3. Data Preparation

###  Preprocessing Strategy
To prevent the neural network from memorizing scanner artifacts or choking on variable dimensions,we implementing an automated OpenCV preprocessing pipeline that executes sequentially and entirely in memory.

*   **Standardization & Cropping:** We will strip erratic color channels to grayscale, use dynamic thresholding to crop away empty margins, and apply an affine warp to mathematically deskew the text.
*   **Geometric Uniformity:** Images will be proportionally resized and zero-padded to a strict 256x256 tensor format without distorting the original letter stroke geometry.
*   **Data Augmentation:** To combat the small dataset size,we embedding a TensorFlow `Sequential` augmentation layer (random rotation, zoom, and brightness) directly into the computational graph. This ensures training data is dynamically warped to prevent memorization, while the validation and test sets remain mathematically pristine.
*   **Partitioning:** We will use Scikit-Learn to enforce a 60/20/20 stratified split, applying balanced class weights to mathematically penalize the network for ignoring the minority class.

## In-Graph Data Augmentation

To prevent the network from memorizing the spatial orientation of our limited dataset, we pass training inputs through an in-graph TensorFlow `Sequential` augmentation layer.
This applies dynamic transformations strictly during the fitting phase:
*   **Random Rotation:** 0.03 (± ~10 degrees) to simulate slightly crooked scans.
*   **Random Zoom:** 0.1 to account for varied camera distances.
*   **Random Brightness:** 0.15 to simulate varied lighting conditions in under-resourced classrooms.

Crucially, because this sits inside the model graph, the validation and test sets remain completely unaltered and mathematically pristine.

In [ ]:

# Preprocessing functions
def standardize_colour(img):
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img

def crop_borders(img, margin=10):
    _, thresh = cv2.threshold(img, 30, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)
    if coords is None:
        return img
    x, y, w, h = cv2.boundingRect(coords)
    x0, y0 = max(x - margin, 0), max(y - margin, 0)
    x1, y1 = min(x + w + margin, img.shape[1]), min(y + h + margin, img.shape[0])
    return img[y0:y1, x0:x1]

def deskew(img):
    _, thresh = cv2.threshold(img, 30, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)
    if coords is None:
        return img
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = 90 + angle
    if abs(angle) < 0.5:
        return img
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), borderValue=0)

def resize_and_pad(img, target_size=TARGET_SIZE):
    h, w = img.shape
    target_w, target_h = target_size
    scale = min(target_w / w, target_h / h)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target_h, target_w), dtype=np.uint8)
    x_off = (target_w - new_w) // 2
    y_off = (target_h - new_h) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized
    return canvas

def normalize(img):
    return img.astype("float32") / 255.0

def preprocess_image(path, target_size=TARGET_SIZE):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    img = standardize_colour(img)
    img = crop_borders(img)
    img = deskew(img)
    img = resize_and_pad(img, target_size)
    img = normalize(img)
    return img

# training-only augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.03),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.15),
], name="data_augmentation")

2026-09-08 15:46:43.772307: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [ ]:
# build a file index
records = []
for cls in CLASSES:
    folder = DATASET_ROOT / cls
    for fname in sorted(os.listdir(folder)):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            records.append({"path": str(folder / fname), "class": cls, "label": LABEL_MAP[cls]})

index_df = pd.DataFrame(records)
print("Total images:", len(index_df))
print(index_df["class"].value_counts())

Total images: 1625
class
Potential Dysgraphia        1049
Low Potential Dysgraphia     576
Name: count, dtype: int64


In [ ]:
# Load and preprocess the images
def load_images(df):
    X = np.stack([preprocess_image(p) for p in df["path"]])
    X = X[.., np.newaxis]  # (N, H, W, 1) — single grayscale channel
    y = df["label"].to_numpy(dtype="float32")
    return X, y

X_all, y_all = load_images(index_df)
print("X_all:", X_all.shape, "y_all:", y_all.shape)

X_all: (1625, 256, 256, 1) y_all: (1625,)


In [ ]:
# 60 / 20 / 20 stratified split
train_idx, temp_idx = train_test_split(
    index_df.index, test_size=0.4, stratify=index_df["label"], random_state=SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, stratify=index_df.loc[temp_idx, "label"], random_state=SEED
)

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val,   y_val   = X_all[val_idx],   y_all[val_idx]
X_test,  y_test  = X_all[test_idx],  y_all[test_idx]

for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name}: n={len(y)}, Potential Dysgraphia={int(y.sum())} ({y.mean():.1%})")

train: n=975, Potential Dysgraphia=629 (64.5%)
val: n=325, Potential Dysgraphia=210 (64.6%)
test: n=325, Potential Dysgraphia=210 (64.6%)


In [ ]:
# Compute class weights to handle class imbalance
class_weights_arr = compute_class_weight(
    class_weight="balanced", classes=np.array([0, 1]), y=y_train
)
class_weight = {0: class_weights_arr[0], 1: class_weights_arr[1]}
print("class_weight:", class_weight)

class_weight: {0: np.float64(1.4089595375722543), 1: np.float64(0.775039745627981)}
